# Music Annotation Project — Notebook 38.6
## Information-Efficient Perceptual Frontier Review

**Runtime:** Colab CPU. Run all cells in order.

**Status:** development-only, single-reviewer exploratory. No retraining, recalibration, threshold adoption, challenger freeze, independent-test nomination or production authority.

### Decision supported
Locate a perceptual sweet-spot **region** on the completed Notebook 37 frontier using a bounded 50–100-comparison review. This replaces exhaustive adjacent-step review with long-baseline, coarse-bracket and breakpoint comparisons.

### Declared deviation
This post-outcome development notebook uses Notebook 37 truth-derived metrics to select high-information cases, caps review burden, centres a very wide crop on actual changed pixels, and does not claim exhaustive D12 compliance. Threshold and truth metrics remain hidden during the blind judgement.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q "gradio>=4.44,<6" pandas numpy pillow matplotlib


## 1. Configuration, helpers and protocol


In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import base64,hashlib,json,os,shutil,zipfile
import numpy as np,pandas as pd
from PIL import Image,ImageDraw
import matplotlib.pyplot as plt
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject');NB37_ROOT=ROOT/'observed_print_classifier_v8_2h_threshold_frontier';OUT_ROOT=ROOT/'observed_print_classifier_v8_2i_envelope_relaxation'
SOURCE_NB37_RUN_OVERRIDE=None;ASSET_ROOT_OVERRIDE=None;VERSION='38.6-information-efficient-frontier';SEED=38006
REVIEW_BUDGET=75;MAX_PER_PAGE=4;MAX_PER_PAIR=15;MAX_CATASTROPHIC=20
WIDE_MIN_W=.55;WIDE_MIN_H=.40;CHANGE_EXPANSION=4.0
THRESHOLDS=[.995,.990,.985,.980,.975,.950,.925,.900,.875,.850,.825,.800,.750,.700,.600,.500]
if not 50<=REVIEW_BUDGET<=100:raise ValueError('REVIEW_BUDGET must be 50–100')
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_2i_38_6';OUT=OUT_ROOT/RUN;TABLES=OUT/'tables';REVIEW=OUT/'review';AUDIT=OUT/'audit';MEDIA=OUT/'review_media';VIS=OUT/'visuals';REF=OUT/'reference';UPLOAD=OUT/'upload'
for p in [OUT,TABLES,REVIEW,AUDIT,MEDIA,VIS,REF]:p.mkdir(parents=True,exist_ok=False)
def now():return datetime.now(timezone.utc).isoformat()
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def text(p,s):p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);q=p.with_name(p.name+'.tmp');q.write_text(s,encoding='utf-8');os.replace(q,p)
def js(p,x):text(p,json.dumps(x,indent=2,sort_keys=True,default=str))
def csv(d,p):p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);q=p.with_name(p.name+'.tmp');d.to_csv(q,index=False);os.replace(q,p)
def stop(stage,msg,details=None):js(AUDIT/f'failure_{stage}.json',{'stage':stage,'message':msg,'details':details or {},'utc':now()});raise RuntimeError(f'{stage}: {msg}')
def col(d,names,required=True):
 z={str(x).lower():x for x in d.columns}
 for n in names:
  if n.lower() in z:return z[n.lower()]
 if required:raise KeyError(f'Missing {names}; columns={list(d.columns)}')
def image_ok(p):
 try:
  with Image.open(p) as im:im.verify()
  return True
 except:return False
def latest(root,required):
 h=[p for p in Path(root).iterdir() if p.is_dir() and all((p/r).exists() for r in required)] if Path(root).exists() else []
 return sorted(h,key=lambda p:p.stat().st_mtime)[-1] if h else None
def sid(prefix,*v):return prefix+'_'+hashlib.sha256('|'.join(map(str,v)).encode()).hexdigest()[:16]
def scale(s):
 s=pd.Series(s,dtype=float);a=s.quantile(.05);b=s.quantile(.95);return ((s-a)/(b-a)).clip(0,1) if b>a else pd.Series(np.zeros(len(s)),index=s.index)
protocol={'version':VERSION,'review_budget':REVIEW_BUDGET,'budget_range':[50,100],'selection_uses_development_truth_metrics':True,'blind_hides_truth_and_threshold':True,'exhaustive_D12_claim':False,'threshold_adoption_authority':False,'production_authority':False,'deviation':'Long-baseline/coarse-bracket information sample replaces exhaustive adjacent-transition review.'};js(OUT/'protocol_lock.json',protocol);print('Output:',OUT)


## 2. Resolve Notebook 37, lock sources and verify frontier


In [ ]:
REQ=['experiment_identity.json','source_v82_record.json','tables/threshold_frontier_summary.csv','tables/incremental_component_review_queue_all.csv','tables/catastrophic_component_first_appearance_inventory.csv','upload/upload_manifest.csv']
NB37=Path(SOURCE_NB37_RUN_OVERRIDE) if SOURCE_NB37_RUN_OVERRIDE else latest(NB37_ROOT,REQ)
if NB37 is None:stop('input','No complete Notebook 37 run found')
locks=[]
for r in REQ:
 p=NB37/r
 if not p.is_file():stop('input','Required file missing',{'file':r})
 locks.append({'relative_path':r,'absolute_path':str(p.resolve()),'bytes':p.stat().st_size,'sha256':sha(p)})
source_lock=pd.DataFrame(locks);csv(source_lock,AUDIT/'notebook38_source_input_lock.csv');source_lock_hash=hashlib.sha256(source_lock.to_csv(index=False).encode()).hexdigest();protocol.update({'source_nb37_run':str(NB37),'source_lock_sha256':source_lock_hash});js(OUT/'protocol_lock.json',protocol);shutil.copy2(NB37/'experiment_identity.json',REF/'experiment_identity.json');shutil.copy2(NB37/'source_v82_record.json',REF/'source_v82_record.json')
front=pd.read_csv(NB37/'tables/threshold_frontier_summary.csv');actual=[round(float(x),3) for x in front[col(front,['threshold'])]]
if actual!=THRESHOLDS:stop('thresholds','Threshold identity/order mismatch',{'actual':actual})
need=['threshold','true_print','false_annotation','utility','catastrophic_components','D3_catastrophic_safety','D4_false_preservation','D5_pixel_safety','D6_positive_utility','D7_nontrivial_benefit','D8_source_stability'];miss=[x for x in need if x not in front]
if miss:stop('frontier','Missing fields',{'missing':miss})
front.threshold=front.threshold.astype(float).round(3);gates=['D3_catastrophic_safety','D4_false_preservation','D5_pixel_safety','D6_positive_utility','D7_nontrivial_benefit','D8_source_stability'];front['gate_count']=front[gates].astype(bool).sum(1);front['all_gates']=front.gate_count.eq(6)
print('Notebook 37:',NB37);print(front[need].to_string(index=False))


## 3. Choose coarse anchors and candidate comparison pairs


In [ ]:
anchors={.995,float(front.loc[front.utility.idxmax(),'threshold'])}
if front.all_gates.any():anchors.add(float(front.loc[front.all_gates,'threshold'].min()))
breaks=front[(front.catastrophic_components.gt(0))|front.gate_count.lt(6)]
if len(breaks):anchors.add(float(breaks.iloc[0].threshold))
for i in np.linspace(0,len(THRESHOLDS)-1,6).round().astype(int):anchors.add(float(THRESHOLDS[i]))
anchors=sorted({round(x,3) for x in anchors if x in THRESHOLDS},reverse=True)
if len(anchors)>8:anchors=sorted({anchors[i] for i in np.linspace(0,len(anchors)-1,8).round().astype(int)},reverse=True)
anchor_table=front[front.threshold.isin(anchors)].copy();csv(anchor_table,TABLES/'selected_threshold_anchors.csv')
pairs=[]
for lo in anchors[1:]:pairs.append({'pair_type':'baseline','hi':.995,'lo':lo})
for hi,lo in zip(anchors[:-1],anchors[1:]):pairs.append({'pair_type':'coarse_bracket','hi':hi,'lo':lo})
central=float(front.loc[front.utility.idxmax(),'threshold']);aggressive=min(anchors)
if central>aggressive:pairs.append({'pair_type':'frontier_stress','hi':central,'lo':aggressive})
if len(breaks):
 bt=float(breaks.iloc[0].threshold);i=THRESHOLDS.index(bt)
 if i>0:pairs.append({'pair_type':'local_breakpoint','hi':THRESHOLDS[i-1],'lo':bt})
pairs=pd.DataFrame(pairs).drop_duplicates(['hi','lo']);csv(pairs,TABLES/'comparison_pair_design.csv');print('Anchors:',anchors);print(pairs.to_string(index=False))


## 4. Build and rank high-information page comparisons


In [ ]:
q=pd.read_csv(NB37/'tables/incremental_component_review_queue_all.csv')
def qc(names,required=True):return col(q,names,required)
tu_col=qc(['new_truth_uncertain','incremental_truth_uncertain','truth_uncertain'],False)
raw=pd.DataFrame({'page':q[qc(['page'])].astype(str),'source_group':q[qc(['source_group'])].astype(str),'component':pd.to_numeric(q[qc(['mask_component_id','component_id'])]).astype(int),'hi':pd.to_numeric(q[qc(['higher_threshold','from_threshold','high_threshold'])]).round(3),'lo':pd.to_numeric(q[qc(['lower_threshold','to_threshold','low_threshold'])]).round(3),'selected':pd.to_numeric(q[qc(['new_selected','new_selected_pixels','incremental_selected','selected'])],errors='coerce').fillna(0),'tp':pd.to_numeric(q[qc(['new_true_print','incremental_true_print','true_print'])],errors='coerce').fillna(0),'fp':pd.to_numeric(q[qc(['new_false_annotation','incremental_false_annotation','false_annotation'])],errors='coerce').fillna(0),'tu':pd.to_numeric(q[tu_col],errors='coerce').fillna(0) if tu_col else 0})
cat=pd.read_csv(NB37/'tables/catastrophic_component_first_appearance_inventory.csv');cp=col(cat,['page']);ct=col(cat,['first_appearance_threshold','threshold','lower_threshold']);cat_keys={(str(r[cp]),round(float(r[ct]),3)) for _,r in cat.iterrows()}
rows=[]
for _,p in pairs.iterrows():
 z=raw[raw.lo.le(float(p.hi)+1e-12)&raw.lo.ge(float(p.lo)-1e-12)]
 for (page,sg),g in z.groupby(['page','source_group']):
  selected=float(g.selected.sum());tp=float(g.tp.sum());fp=float(g.fp.sum());tu=float(g.tu.sum())
  if selected<=0:continue
  catastrophic=any((str(page),round(float(t),3)) in cat_keys for t in g.lo.unique())
  rows.append({'page':page,'source_group':sg,'pair_type':p.pair_type,'hi':float(p.hi),'lo':float(p.lo),'span':float(p.hi-p.lo),'selected':selected,'tp':tp,'fp':fp,'tu':tu,'utility':tp-5*fp,'components':g.component.nunique(),'catastrophic':catastrophic})
candidates=pd.DataFrame(rows)
if candidates.empty:stop('candidate_pool','No comparisons constructed')
candidates['s_selected']=scale(np.log1p(candidates.selected));candidates['s_fp']=scale(np.log1p(candidates.fp));candidates['s_tp']=scale(np.log1p(candidates.tp));candidates['s_components']=scale(np.log1p(candidates.components));candidates['s_span']=scale(candidates.span);candidates['s_tradeoff']=np.minimum(candidates.s_fp,candidates.s_tp)
candidates['information_score']=1.2*candidates.s_selected+1.8*candidates.s_fp+1.5*candidates.s_tp+candidates.s_components+1.2*candidates.s_span+1.5*candidates.s_tradeoff+4*candidates.catastrophic.astype(float)
candidates['comparison_id']=candidates.apply(lambda r:sid('CMP',r.page,r.hi,r.lo),axis=1);csv(candidates.sort_values('information_score',ascending=False),TABLES/'comparison_candidate_pool.csv')
print('Candidate comparisons:',len(candidates))


## 5. Select a bounded, diverse queue


In [ ]:
chosen=[];page_n={};pair_n={}
def can_add(r):return page_n.get(str(r.page),0)<MAX_PER_PAGE and pair_n.get((r.hi,r.lo),0)<MAX_PER_PAIR and r.comparison_id not in chosen
def take(frame,n):
 k=0
 for _,r in frame.iterrows():
  if len(chosen)>=REVIEW_BUDGET or k>=n:break
  if can_add(r):chosen.append(r.comparison_id);page_n[str(r.page)]=page_n.get(str(r.page),0)+1;pair_n[(r.hi,r.lo)]=pair_n.get((r.hi,r.lo),0)+1;k+=1
def quota(x):return max(1,int(round(REVIEW_BUDGET*x)))
take(candidates[candidates.catastrophic].sort_values('information_score',ascending=False),min(MAX_CATASTROPHIC,quota(.20)));take(candidates.sort_values(['s_fp','information_score'],ascending=False),quota(.20));take(candidates.sort_values(['s_tp','information_score'],ascending=False),quota(.20));take(candidates.sort_values(['s_tradeoff','information_score'],ascending=False),quota(.18));take(candidates.sort_values('information_score',ascending=False).drop_duplicates(['source_group','hi','lo']),quota(.14));take(candidates.sort_values(['s_selected','information_score'],ascending=[True,False]).drop_duplicates('source_group'),quota(.08));take(candidates.sort_values('information_score',ascending=False),REVIEW_BUDGET)
queue=candidates[candidates.comparison_id.isin(chosen)].copy();queue['selection_order']=queue.comparison_id.map({x:i for i,x in enumerate(chosen)});queue=queue.sort_values('selection_order').reset_index(drop=True)
if len(queue)<min(50,len(candidates)):stop('queue','Fewer than 50 comparisons under caps',{'selected':len(queue),'candidates':len(candidates)})
csv(queue,TABLES/'selected_information_review_queue_unblinded.csv');coverage=queue.groupby(['hi','lo','pair_type'],as_index=False).agg(comparisons=('comparison_id','size'),pages=('page','nunique'),source_groups=('source_group','nunique'),tp=('tp','sum'),fp=('fp','sum'));csv(coverage,TABLES/'review_queue_coverage.csv');print('Selected:',len(queue),'pages:',queue.page.nunique(),'groups:',queue.source_group.nunique());print(coverage.to_string(index=False))


## 6. Resolve Notebook 37 persisted threshold pages


In [ ]:
N={'candidate':'threshold_candidate.png','registered':'v8_2_registered.png','v78':'v7_8.png','observed':'observed.png','difference':'difference_from_registered_v8_2.png','overlay':'newly_preserved_vs_0995_overlay.png'}
ASSET_ROOT=Path(ASSET_ROOT_OVERRIDE) if ASSET_ROOT_OVERRIDE else NB37;all_dirs=sorted({p.parent.resolve() for p in ASSET_ROOT.rglob(N['candidate'])});working=[d for d in all_dirs if not (NB37/'upload').resolve() in d.parents];dirs=working if working else all_dirs
if not dirs:stop('assets','No threshold_candidate.png found')
def palias(page):
 s=str(page);d=''.join(x for x in s if x.isdigit());a={s.lower(),Path(s).stem.lower()}
 if d:
  n=int(d);a|={str(n),f'{n:02d}',f'{n:03d}',f'page{n}',f'page{n:02d}',f'page{n:03d}',f'page_{n:02d}',f'page_{n:03d}'}
 return a
def talias(t):
 s=f'{float(t):.3f}';return {s,s.replace('.','_'),s.replace('.',''),f't{s}',f't_{s}',f'threshold_{s}',f'threshold_{s.replace(".","_")}'}
def resolve(page,t):
 ranked=[]
 for d in dirs:
  parts=[x.lower() for x in d.parts];joined='/'.join(parts);ps=max([4 if a in parts else 2 if a in joined else 0 for a in palias(page)] or [0]);ts=max([5 if a in parts else 3 if a in joined else 0 for a in talias(t)] or [0])
  if ps and ts:ranked.append((ps+ts,d))
 if not ranked:return None,'no page/threshold match',[]
 ranked=sorted(ranked,key=lambda x:(x[0],-len(x[1].parts)),reverse=True);top=ranked[0][0];tops=sorted({d for s,d in ranked if s==top},key=str)
 return (tops[0],'',ranked[:5]) if len(tops)==1 else (None,f'ambiguous top score {top}',ranked[:5])
def get(d,name):
 p=Path(d)/name if d else None;return p if p and p.is_file() and image_ok(p) else None
resolved=[];audit=[];fails=[]
for _,r in queue.iterrows():
 ld,le,lr=resolve(r.page,r.lo);hd,he,hr=(ld,'',[]) if np.isclose(r.hi,.995) else resolve(r.page,r.hi);low=get(ld,N['candidate']);high=get(ld,N['registered']) if np.isclose(r.hi,.995) else get(hd,N['candidate']);reasons=[x for x in [le,he] if x]
 if not low:reasons.append('lower missing')
 if not high:reasons.append('higher missing')
 if low and high:
  with Image.open(low) as a,Image.open(high) as b:
   if a.size!=b.size:reasons.append('shape mismatch')
 ar={'comparison_id':r.comparison_id,'page':r.page,'hi':r.hi,'lo':r.lo,'low_path':str(low or ''),'high_path':str(high or ''),'status':'failed' if reasons else 'verified','reason':' | '.join(reasons)};audit.append(ar)
 if reasons:fails.append(ar);continue
 diag={k:get(ld,v) for k,v in N.items() if k!='candidate'};resolved.append({**r.to_dict(),'low_path':str(low),'high_path':str(high),'overlay':str(diag.get('overlay') or ''),'observed':str(diag.get('observed') or ''),'v78':str(diag.get('v78') or '')})
csv(pd.DataFrame(audit),AUDIT/'whole_page_asset_resolution.csv');csv(pd.DataFrame(fails),AUDIT/'missing_review_assets.csv')
if fails:stop('assets','Selected asset resolution failed',{'count':len(fails),'examples':fails[:10]})
resolved=pd.DataFrame(resolved);print('Resolved:',len(resolved))


## 7. Generate wide changed-region context and blinded A/B

The primary crop contains all changed pixels, spans at least 55% of page width and 40% of page height, and is at least four times the changed bounding box. A page locator and optional full pages provide broader context. The exact difference is hidden until the first judgement is saved.


In [ ]:
def bounds(mask,w,h):
 ys,xs=np.where(mask)
 if len(xs)==0:return None
 x0,x1=int(xs.min()),int(xs.max())+1;y0,y1=int(ys.min()),int(ys.max())+1;bw=max(1,x1-x0);bh=max(1,y1-y0);tw=min(w,max(int(w*WIDE_MIN_W),int(bw*CHANGE_EXPANSION)));th=min(h,max(int(h*WIDE_MIN_H),int(bh*CHANGE_EXPANSION)));cx=(x0+x1)//2;cy=(y0+y1)//2;left=max(0,min(w-tw,cx-tw//2));top=max(0,min(h-th,cy-th//2));return left,top,left+tw,top+th,(x0,y0,x1,y1)
def media_for(r):
 hi=Image.open(r.high_path).convert('L');lo=Image.open(r.low_path).convert('L');a=np.array(hi);b=np.array(lo);changed=a!=b;z=bounds(changed,hi.width,hi.height)
 if z is None:return None
 left,top,right,bottom,box=z;d=MEDIA/r.comparison_id;d.mkdir(parents=True,exist_ok=True);hp=d/'high_context.png';lp=d/'low_context.png';hi.crop((left,top,right,bottom)).save(hp);lo.crop((left,top,right,bottom)).save(lp);locator=hi.convert('RGB');draw=ImageDraw.Draw(locator);draw.rectangle(box,outline=(230,0,0),width=max(4,int(max(locator.size)/500)));loc=d/'page_locator.png';locator.save(loc);arr=np.array(hi.convert('RGB'));arr[changed]=[230,0,0];diff=d/'exact_difference.png';Image.fromarray(arr).save(diff);hi.close();lo.close();return {'high_context':str(hp),'low_context':str(lp),'locator':str(loc),'exact_difference':str(diff),'crop_box':json.dumps([left,top,right,bottom]),'change_box':json.dumps(list(box)),'changed_pixels':int(changed.sum())}
rows=[];same=[]
for _,r in resolved.iterrows():
 m=media_for(r)
 if m is None:same.append({'comparison_id':r.comparison_id,'reason':'identical endpoints'})
 else:rows.append({'comparison_id':r.comparison_id,**m})
csv(pd.DataFrame(same),AUDIT/'identical_endpoint_exclusions.csv');resolved=resolved.merge(pd.DataFrame(rows),on='comparison_id',validate='one_to_one');blind=[]
for _,r in resolved.iterrows():
 low_left=int(hashlib.sha256(f'{SEED}|{r.comparison_id}'.encode()).hexdigest(),16)%2==0;Ac,Bc=(r.low_context,r.high_context) if low_left else (r.high_context,r.low_context);Af,Bf=(r.low_path,r.high_path) if low_left else (r.high_path,r.low_path);side={'A':'lower','B':'higher'} if low_left else {'A':'higher','B':'lower'};blind.append({'review_unit_id':sid('RV',r.comparison_id),'comparison_id':r.comparison_id,'A_context':Ac,'B_context':Bc,'A_full':Af,'B_full':Bf,'locator':r.locator,'exact_difference':r.exact_difference,'overlay':r.overlay,'observed':r.observed,'v78':r.v78,'side_lock':base64.urlsafe_b64encode(json.dumps(side).encode()).decode(),'changed_pixels':r.changed_pixels})
blind=pd.DataFrame(blind);csv(blind,TABLES/'blinded_information_review_manifest.csv');csv(resolved,TABLES/'selected_review_queue_unblinded_resolved.csv');print('Final queue:',len(blind))


## 8. Focused Gradio reviewer

Review only the changed region. No forced A/B preference is required. “No material difference” is a complete answer. After the blind judgement is locked, diagnostic imagery appears.


In [ ]:
import gradio as gr
R1=REVIEW/'focused_review_round1.csv';R2=REVIEW/'focused_review_diagnostic.csv';C1=['review_unit_id','material_difference','notation_benefit','annotation_harm','overall','scope','confidence','note','utc','locked','record_sha256'];C2=['review_unit_id','mechanism','final_harm','final_benefit','note','utc','record_sha256']
def readr(p,cols):
 if not Path(p).exists():return pd.DataFrame(columns=cols)
 d=pd.read_csv(p,keep_default_na=False)
 for x in cols:
  if x not in d:d[x]=''
 return d[cols]
def save(p,cols,d):
 d['record_sha256']=hashlib.sha256(json.dumps(d,sort_keys=True,default=str).encode()).hexdigest();z=readr(p,cols);z=z[z.review_unit_id.astype(str)!=str(d['review_unit_id'])];csv(pd.concat([z,pd.DataFrame([d])],ignore_index=True)[cols],p);return d['record_sha256']
def pending():
 done=set(readr(R2,C2).review_unit_id.astype(str));return [i for i,x in enumerate(blind.review_unit_id.astype(str)) if x not in done]
def im(x):return str(x) if str(x) and Path(str(x)).is_file() else None
def view(i):
 i=int(max(0,min(int(i),len(blind)-1)));r=blind.iloc[i];return i,im(r.A_context),im(r.B_context),im(r.locator),im(r.A_full),im(r.B_full),f'### Focused comparison {i+1} of {len(blind)}  \nCompleted: {len(readr(R2,C2))}  \nThe wide crop contains every changed pixel.',r.review_unit_id
def first(i,material,benefit,harm,overall,scope,confidence,note):
 if any(x in ['',None] for x in [material,benefit,harm,overall,scope,confidence]):raise gr.Error('Complete every first-pass question.')
 r=blind.iloc[int(i)];h=save(R1,C1,{'review_unit_id':r.review_unit_id,'material_difference':material,'notation_benefit':benefit,'annotation_harm':harm,'overall':overall,'scope':scope,'confidence':confidence,'note':str(note),'utc':now(),'locked':True});return f'Blind judgement locked ({h[:12]}).',gr.update(visible=False),gr.update(visible=True),im(r.exact_difference),im(r.overlay),im(r.observed),im(r.v78)
def diagnostic(i,mechanism,harm,benefit,note):
 if any(x in ['',None] for x in [mechanism,harm,benefit]):raise gr.Error('Complete all diagnostic questions.')
 r=blind.iloc[int(i)];h=save(R2,C2,{'review_unit_id':r.review_unit_id,'mechanism':mechanism,'final_harm':harm,'final_benefit':benefit,'note':str(note),'utc':now()});p=pending();nxt=next((x for x in p if x>int(i)),p[0] if p else int(i));return view(nxt)+(f'Saved ({h[:12]}). Loading {nxt+1}.',gr.update(visible=True),gr.update(visible=False))
GUIDE='## Review only what changed\n- The wide crops use identical coordinates and contain every changed pixel.\n- The locator shows that region on the full page.\n- Use full pages only for musical context; do not compare unrelated areas.\n- You are not required to prefer A or B.\n- Choose No material difference when appropriate.'
with gr.Blocks(title='Notebook 38.6 focused frontier review') as app:
 gr.Markdown('# Notebook 38.6 — information-efficient focused review');gr.Markdown(GUIDE);state=gr.State(pending()[0] if pending() else 0);unit=gr.Textbox(visible=False);status=gr.Markdown()
 with gr.Row():A=gr.Image(type='filepath',label='Wide context A',height=650);B=gr.Image(type='filepath',label='Wide context B',height=650)
 locator=gr.Image(type='filepath',label='Page locator — red box is the changed region',height=450)
 with gr.Accordion('Optional full-page context',open=False):
  with gr.Row():Afull=gr.Image(type='filepath',label='Full page A');Bfull=gr.Image(type='filepath',label='Full page B')
 with gr.Column(visible=True) as p1:
  gr.Markdown('## Blind first pass');material=gr.Radio(['No material difference','Material difference','Cannot tell'],label='1. Is the changed region perceptually meaningful?');benefit=gr.Radio(['Neither','A','B','Both differently','Cannot tell'],label='2. Does either preserve genuine notation better?');harm=gr.Radio(['Neither','A','B','Both differently','Cannot tell'],label='3. Does either retain more objectionable annotation?');overall=gr.Radio(['No material difference','A better overall','B better overall','Trade-off','Both unacceptable','Cannot tell'],label='4. Overall assessment');scope=gr.Dropdown(['Pixel/fragment','One symbol/component','Part of one staff/system','Multiple systems','Page-level','Cannot tell'],label='5. Largest meaningful scope');confidence=gr.Radio(['Low','Medium','High'],label='6. Confidence');note=gr.Textbox(label='7. Optional note',lines=3);b1=gr.Button('Save blind judgement and reveal diagnostics',variant='primary')
 with gr.Column(visible=False) as p2:
  gr.Markdown('## Diagnostic pass — first pass locked')
  with gr.Row():exact=gr.Image(label='Exact A/B difference');overlay=gr.Image(label='Notebook 37 overlay');observed=gr.Image(label='Observed');v78=gr.Image(label='Frozen V7.8')
  mechanism=gr.Dropdown(['Genuine print recovery','Retained annotation','Mixed print and annotation','Registration/truth ambiguity','No material difference','Display artefact','Other'],label='8. Best explanation');fh=gr.Radio(['P0','P1','P2','P3','P4','U'],label='9. Final harm: P0 none; P4 catastrophic');fb=gr.Radio(['B0','B1','B2','B3','B4','U'],label='10. Final benefit: B0 none; B4 page-level');dn=gr.Textbox(label='11. Diagnostic note',lines=3);b2=gr.Button('Save diagnostic and load next comparison',variant='primary')
 message=gr.Markdown();app.load(view,[state],[state,A,B,locator,Afull,Bfull,status,unit]);b1.click(first,[state,material,benefit,harm,overall,scope,confidence,note],[message,p1,p2,exact,overlay,observed,v78]);b2.click(diagnostic,[state,mechanism,fh,fb,dn],[state,A,B,locator,Afull,Bfull,status,unit,message,p1,p2])
app.launch(share=True,debug=False)


## 9. Completion, direct-pair analysis and descriptive sweet-spot


In [ ]:
r1=readr(R1,C1);r2=readr(R2,C2);expected=set(blind.review_unit_id.astype(str));d1=set(r1.review_unit_id.astype(str));d2=set(r2.review_unit_id.astype(str));coverage=pd.DataFrame({'review_unit_id':sorted(expected),'round1_complete':[x in d1 for x in sorted(expected)],'diagnostic_complete':[x in d2 for x in sorted(expected)]});csv(coverage,AUDIT/'review_coverage.csv')
if not coverage[['round1_complete','diagnostic_complete']].all().all():stop('completion','Review incomplete',{'expected':len(expected),'round1':len(d1&expected),'diagnostic':len(d2&expected)})
completed=blind.merge(r1,on='review_unit_id',validate='one_to_one').merge(r2,on='review_unit_id',validate='one_to_one').merge(resolved[['comparison_id','page','source_group','pair_type','hi','lo','tp','fp','utility','information_score']],on='comparison_id',validate='one_to_one');completed['side_mapping']=completed.side_lock.map(lambda s:json.dumps(json.loads(base64.urlsafe_b64decode(str(s).encode()).decode()),sort_keys=True));csv(completed,TABLES/'completed_focused_comparisons.csv')
summary=completed.groupby(['hi','lo','pair_type'],as_index=False).agg(reviews=('review_unit_id','size'),pages=('page','nunique'),source_groups=('source_group','nunique'),material_rate=('material_difference',lambda s:float((s=='Material difference').mean())),no_material_rate=('material_difference',lambda s:float((s=='No material difference').mean())),tradeoff_rate=('overall',lambda s:float((s=='Trade-off').mean())),P2_plus_rate=('final_harm',lambda s:float(s.isin(['P2','P3','P4']).mean())),P3_plus_count=('final_harm',lambda s:int(s.isin(['P3','P4']).sum())),B2_plus_rate=('final_benefit',lambda s:float(s.isin(['B2','B3','B4']).mean())),B2_plus_count=('final_benefit',lambda s:int(s.isin(['B2','B3','B4']).sum())));csv(summary,TABLES/'direct_perceptual_pair_summary.csv');baseline=summary[np.isclose(summary.hi,.995)].sort_values('lo',ascending=False);csv(baseline,TABLES/'baseline_vs_candidate_perceptual_frontier.csv');eligible=baseline[(baseline.P3_plus_count.eq(0))&(baseline.B2_plus_count.gt(0))];sweet=float(eligible.lo.min()) if len(eligible) else None
verdict={'version':VERSION,'status':'single-reviewer-exploratory-information-efficient','completed_comparisons':len(completed),'review_budget':REVIEW_BUDGET,'baseline_candidates_reviewed':baseline.lo.tolist(),'descriptive_sweet_spot_candidate':sweet,'exhaustive_D12_claim':False,'threshold_adoption_authority':False,'production_authority':False,'interpretation':'Direct selected comparisons only; no transitive equivalence or promotion claim.'};js(OUT/'notebook38_verdict.json',verdict);text(OUT/'executive_summary.txt',json.dumps(verdict,indent=2)+'\n');print(summary.to_string(index=False));print(json.dumps(verdict,indent=2))


## 10. Visual summary and final package


In [ ]:
if len(baseline):
 plt.figure(figsize=(9,5));plt.plot(baseline.lo,baseline.B2_plus_rate,marker='o',label='B2+ benefit');plt.plot(baseline.lo,baseline.P2_plus_rate,marker='o',label='P2+ harm');plt.gca().invert_xaxis();plt.xlabel('Candidate threshold directly compared with 0.995');plt.ylabel('Reviewed rate');plt.legend();plt.tight_layout();plt.savefig(VIS/'baseline_candidate_frontier.png',dpi=160);plt.close()
items=[OUT/'protocol_lock.json',OUT/'notebook38_verdict.json',OUT/'executive_summary.txt',TABLES/'selected_threshold_anchors.csv',TABLES/'comparison_pair_design.csv',TABLES/'comparison_candidate_pool.csv',TABLES/'selected_information_review_queue_unblinded.csv',TABLES/'review_queue_coverage.csv',TABLES/'blinded_information_review_manifest.csv',TABLES/'selected_review_queue_unblinded_resolved.csv',TABLES/'completed_focused_comparisons.csv',TABLES/'direct_perceptual_pair_summary.csv',TABLES/'baseline_vs_candidate_perceptual_frontier.csv',R1,R2,AUDIT/'notebook38_source_input_lock.csv',AUDIT/'whole_page_asset_resolution.csv',AUDIT/'missing_review_assets.csv',AUDIT/'identical_endpoint_exclusions.csv',AUDIT/'review_coverage.csv',REF/'experiment_identity.json',REF/'source_v82_record.json']
if (VIS/'baseline_candidate_frontier.png').exists():items.append(VIS/'baseline_candidate_frontier.png')
missing=[str(x) for x in items if not x.exists()]
if missing:stop('package','Missing outputs',{'missing':missing})
UPLOAD.mkdir();rows=[]
for src in items:
 rel=src.relative_to(OUT);dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(src,dst);rows.append({'relative_path':rel.as_posix(),'bytes':dst.stat().st_size,'sha256':sha(dst)})
text(UPLOAD/'README.txt','Notebook 38.6 information-efficient post-outcome review; not exhaustive D12 or threshold adoption evidence.\n');rows.append({'relative_path':'README.txt','bytes':(UPLOAD/'README.txt').stat().st_size,'sha256':sha(UPLOAD/'README.txt')});csv(pd.DataFrame(rows),UPLOAD/'upload_manifest.csv');bundle=OUT/'notebook38_6_information_review_bundle.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
 for p in UPLOAD.rglob('*'):
  if p.is_file():z.write(p,arcname=(Path('upload')/p.relative_to(UPLOAD)).as_posix())
js(OUT/'final_package_record.json',{'bundle':str(bundle),'sha256':sha(bundle),'utc':now()});print(bundle)


## Interpretation boundary
Notebook 38.6 asks which separated threshold regions produce visible benefit, visible harm, trade-offs or no material difference on a selected high-information development sample. It does not prove complete-population robustness, satisfy exhaustive D12, infer global equivalence from adjacent steps or authorise threshold adoption.
